# FINANCE 384 Assignment 1 – Part A

## Task A.2: Pooled OLS Benchmark

This standalone notebook estimates the transparent pooled ordinary least squares (OLS) benchmark required for Task A.2:

\[
r^e_{i,t+1}=\alpha+\beta'z_{i,t}+\varepsilon_{i,t+1}.
\]

It rebuilds the minimum data setup needed for A.2, fits OLS on the **combined training + validation sample**, and generates untouched test-period forecasts for later use in A.5 and A.6.


### Files required

Upload these files into the Colab/Jupyter working directory before running:

- `FINANCE384_assignmentA_development_panel.csv`
- `FINANCE384_stock_month_data_dictionary.csv`

`FINANCE384_market.csv` is not required for A.2.


In [1]:
# A.2.1 Imports and file paths
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

PANEL_FILE = "FINANCE384_assignmentA_development_panel.csv"
DICTIONARY_FILE = "FINANCE384_stock_month_data_dictionary.csv"


In [2]:
# A.2.2 Load the supplied files
panel = pd.read_csv(PANEL_FILE)
data_dictionary = pd.read_csv(DICTIONARY_FILE)

panel["date"] = pd.to_datetime(panel["date"])

print("Panel shape:", panel.shape)
print("Date range:", panel["date"].min().date(), "to", panel["date"].max().date())
print("Unique stocks:", panel["permno"].nunique())
print("Unique months:", panel["date"].dt.to_period("M").nunique())
print("Duplicate stock-month rows:", panel.duplicated(["permno", "date"]).sum())


Panel shape: (198298, 27)
Date range: 1990-01-31 to 2022-12-30
Unique stocks: 1252
Unique months: 396
Duplicate stock-month rows: 0


### Base predictor information

The benchmark uses the same base information planned for the richer model: 18 numeric stock/market characteristics plus FF49 industry membership. Identifiers and realised outcome variables are excluded.


In [3]:
# A.2.3 Define predictors explicitly
numeric_predictors = [
    "size", "bm", "mom12_2", "vol12", "beta60", "ivol60",
    "turnover", "dollar_volume", "amihud_illiq", "divyield",
    "gross_profit", "roe", "asset_growth", "leverage", "accruals",
    "mkt_12m", "mkt_vol_12m", "down_market",
]

continuous_predictors = [x for x in numeric_predictors if x != "down_market"]
binary_predictors = ["down_market"]
categorical_predictors = ["ff49_code"]
feature_columns = continuous_predictors + binary_predictors + categorical_predictors

print("Numeric predictors:", len(numeric_predictors))
print("Categorical predictor:", categorical_predictors)
print("Total raw feature columns:", len(feature_columns))


Numeric predictors: 18
Categorical predictor: ['ff49_code']
Total raw feature columns: 19


### Construct the one-month-ahead target

The next return is retained only when the following observation for the same `permno` is exactly one calendar month later. This avoids incorrectly linking observations across gaps in index membership.


In [4]:
# A.2.4 Construct the next-month target
analysis = panel.sort_values(["permno", "date"]).copy()

analysis["next_date"] = analysis.groupby("permno")["date"].shift(-1)
analysis["ret_excess_t1"] = analysis.groupby("permno")["ret_excess_t"].shift(-1)

analysis["is_consecutive_next_month"] = (
    analysis["next_date"].dt.to_period("M")
    == analysis["date"].dt.to_period("M") + 1
)

analysis.loc[~analysis["is_consecutive_next_month"], "ret_excess_t1"] = np.nan

gapped_observations = (
    analysis["next_date"].notna() & ~analysis["is_consecutive_next_month"]
).sum()

analysis_valid = analysis.loc[analysis["ret_excess_t1"].notna()].copy()

print("Raw stock-month rows:", len(analysis))
print("Non-consecutive next observations detected:", int(gapped_observations))
print("Rows with valid next-month targets:", len(analysis_valid))


Raw stock-month rows: 198298
Non-consecutive next observations detected: 30
Rows with valid next-month targets: 197016


### Fixed chronological samples

The split is based on forecast decision date \(t\). OLS is estimated on training + validation, while the test period is reserved for out-of-sample prediction.


In [5]:
# A.2.5 Apply the required sample periods
train = analysis_valid.loc[
    (analysis_valid["date"] >= "1990-01-01")
    & (analysis_valid["date"] <= "2014-12-31")
].copy()

validation = analysis_valid.loc[
    (analysis_valid["date"] >= "2015-01-01")
    & (analysis_valid["date"] <= "2018-12-31")
].copy()

test = analysis_valid.loc[
    (analysis_valid["date"] >= "2019-01-01")
    & (analysis_valid["date"] <= "2022-11-30")
].copy()

ols_estimation = pd.concat([train, validation], axis=0).sort_values(
    ["date", "permno"]
).reset_index(drop=True)

split_summary = pd.DataFrame({
    "Sample": ["Training", "Validation", "OLS estimation", "Test"],
    "Decision period": [
        "Jan 1990-Dec 2014",
        "Jan 2015-Dec 2018",
        "Jan 1990-Dec 2018",
        "Jan 2019-Nov 2022",
    ],
    "Months": [
        train["date"].dt.to_period("M").nunique(),
        validation["date"].dt.to_period("M").nunique(),
        ols_estimation["date"].dt.to_period("M").nunique(),
        test["date"].dt.to_period("M").nunique(),
    ],
    "Stock-month rows": [len(train), len(validation), len(ols_estimation), len(test)],
})

split_summary


,Sample,Decision period,Months,Stock-month rows
0,Training,Jan 1990-Dec 2014,300,149334
1,Validation,Jan 2015-Dec 2018,48,24051
2,OLS estimation,Jan 1990-Dec 2018,348,173385
3,Test,Jan 2019-Nov 2022,47,23631


### Preprocessing rule

- Continuous predictors: training-sample median imputation, then training-sample standardisation.
- `down_market`: most-frequent imputation only; remains 0/1.
- `ff49_code`: one-hot encoded with one reference category omitted.
- Test-period information is not used to estimate preprocessing parameters.


In [6]:
# A.2.6 Define and fit preprocessing on training data
continuous_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

categorical_pipeline = Pipeline(steps=[
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", continuous_pipeline, continuous_predictors),
        ("binary", binary_pipeline, binary_predictors),
        ("industry", categorical_pipeline, categorical_predictors),
    ],
    remainder="drop",
)

X_train_raw = train[feature_columns]
X_ols_estimation_raw = ols_estimation[feature_columns]
X_test_raw = test[feature_columns]

y_ols_estimation = ols_estimation["ret_excess_t1"]
y_test = test["ret_excess_t1"]

preprocessor.fit(X_train_raw)

X_ols_estimation = preprocessor.transform(X_ols_estimation_raw)
X_test = preprocessor.transform(X_test_raw)

print("OLS estimation matrix:", X_ols_estimation.shape)
print("Test matrix:", X_test.shape)


OLS estimation matrix: (173385, 64)
Test matrix: (23631, 64)


In [7]:
# A.2.7 Estimate pooled OLS
ols_model = LinearRegression(fit_intercept=True)
ols_model.fit(X_ols_estimation, y_ols_estimation)

print("OLS fitted successfully.")
print("Intercept:", round(float(ols_model.intercept_), 6))
print("Number of slope coefficients:", len(ols_model.coef_))


OLS fitted successfully.
Intercept: 0.010555
Number of slope coefficients: 64


### Generate test-period forecasts

The fitted OLS benchmark is applied to the untouched test feature matrix. The resulting predictions are retained for A.5 and A.6.


In [8]:
# A.2.8 Generate test predictions
ols_test_pred = ols_model.predict(X_test)

ols_test_predictions = test[
    ["date", "permno", "ticker", "ret_excess_t1"]
].copy()

ols_test_predictions = ols_test_predictions.rename(
    columns={"ret_excess_t1": "actual_excess_return_t1"}
)

ols_test_predictions["ols_pred_excess_return_t1"] = ols_test_pred

print("Test predictions generated:", len(ols_test_predictions))
print(
    "Missing test predictions:",
    int(ols_test_predictions["ols_pred_excess_return_t1"].isna().sum())
)
print(
    "Prediction date range:",
    ols_test_predictions["date"].min().date(),
    "to",
    ols_test_predictions["date"].max().date()
)

ols_test_predictions.head()


Test predictions generated: 23631
Missing test predictions: 0
Prediction date range: 2019-01-31 to 2022-11-30


,date,permno,ticker,actual_excess_return_t1,ols_pred_excess_return_t1
587,2019-01-31,10104,ORCL,0.036026,0.001409
588,2019-02-28,10104,ORCL,0.028409,0.008911
589,2019-03-29,10104,ORCL,0.032531,0.006138
590,2019-04-30,10104,ORCL,-0.087587,0.011721
591,2019-05-31,10104,ORCL,0.124089,0.012534


### Coefficient audit

The coefficient table is retained as a model-audit output rather than as a final performance result.


In [9]:
# A.2.9 Coefficient audit table
feature_names = preprocessor.get_feature_names_out()

ols_coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": ols_model.coef_,
})

ols_coefficients["abs_coefficient"] = ols_coefficients["coefficient"].abs()

ols_coefficients_sorted = (
    ols_coefficients
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

ols_coefficients_sorted.head(15)


,feature,coefficient,abs_coefficient
0,industry__ff49_code_29,-0.013954,0.013954
1,industry__ff49_code_20,-0.012699,0.012699
2,industry__ff49_code_16,-0.012397,0.012397
3,industry__ff49_code_7,0.010955,0.010955
4,industry__ff49_code_31,-0.007128,0.007128
5,continuous__bm,0.007108,0.007108
6,industry__ff49_code_27,-0.006576,0.006576
7,industry__ff49_code_36,0.006083,0.006083
8,industry__ff49_code_12,0.005596,0.005596
9,continuous__mkt_vol_12m,0.005540,0.005540


In [10]:
# A.2.10 Final audit checks
ols_audit = pd.DataFrame({
    "Item": [
        "OLS estimation observations",
        "Test observations",
        "Transformed predictors",
        "Missing test predictions",
        "Mean test prediction",
        "Std. dev. test prediction",
    ],
    "Value": [
        len(ols_estimation),
        len(test),
        X_ols_estimation.shape[1],
        int(ols_test_predictions["ols_pred_excess_return_t1"].isna().sum()),
        float(ols_test_predictions["ols_pred_excess_return_t1"].mean()),
        float(ols_test_predictions["ols_pred_excess_return_t1"].std()),
    ],
})

ols_audit


,Item,Value
0,OLS estimation observations,173385.000000
1,Test observations,23631.000000
2,Transformed predictors,64.000000
3,Missing test predictions,0.000000
4,Mean test prediction,0.010656
5,Std. dev. test prediction,0.011403


## A.2 Summary

The pooled OLS benchmark is estimated on the combined training and validation sample and produces one next-month excess-return forecast for every valid test stock-month observation.

Formal benchmark evaluation is intentionally deferred to **A.5** (RMSE and mean monthly Spearman rank correlation) and **A.6** (quintile portfolios, P5−P1 return, t-statistic, and market-model alpha).
